# Attention Is All You Need 재구현 실습 - 추가 실습 코드: Causal (Masked) Attention 구현

- Tutorial ID: `mod-attention-paper-lab`
- Tutorial: Attention Is All You Need 재구현 실습
- Section ID: `practice-attention-mask`
- Section: 추가 실습 코드: Causal (Masked) Attention 구현


# Integrated Notebook: 03_Causal_Masked_Attention

## Original Version
Source: 065_mod-attention-paper-lab_practice-attention-mask_Attention_Is_All_You_Need_#Uc7ac#Uad6c#Ud604_#Uc2e4#Uc2b5_-_#Ucd94#Uac00_#Uc2e4#Uc2b5_#Ucf54#Ub4dc-_Causal_(Masked)_Attention_#Uad6c#Ud604.ipynb

# Attention Is All You Need 재구현 실습 - 추가 실습 코드: Causal (Masked) Attention 구현

- Tutorial ID: `mod-attention-paper-lab`
- Tutorial: Attention Is All You Need 재구현 실습
- Section: 추가 실습 코드: Causal (Masked) Attention 구현

In [ ]:
# ============================================================
# 코드 읽는 법 — 추가 실습 코드: Causal (Masked) Attention 구현
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def create_causal_mask(seq_len):
    """
    Causal (Look-ahead) 마스크 생성
    - 미래 토큰을 볼 수 없게 하는 하삼각 행렬
    """
    # 하삼각 행렬: 현재 및 과거 위치만 1
        mask = np.tril(np.ones((seq_len, seq_len)))
    return mask

def masked_attention(Q, K, V, mask=None):
    """Causal Attention 구현"""
    d_k = Q.shape[-1]
    
    # QK^T / sqrt(d_k)
        scores = np.matmul(Q, K.T) / np.sqrt(d_k)
    
    # 마스크 적용: 0인 위치에 -inf
    if mask is not None:
                scores = np.where(mask == 0, -1e9, scores)
        print("마스킹된 scores (미래 토큰은 -inf):")
        print(np.where(scores < -1e8, "-∞", scores.round(2)))
    
    # Softmax
        weights = softmax(scores)
    print("\\nAttention weights (미래 토큰의 확률 = 0):")
    print(weights.round(3))
    
        output = np.matmul(weights, V)
    return output, weights

# 테스트
np.random.seed(42)
seq_len, d_k = 5, 4

Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

print("=" * 50)
print("Causal (Masked) Attention for Auto-regressive LM")
print("=" * 50)

# 토큰 이름
tokens = ["I", "love", "machine", "learning", "!"]

print("\\n입력 토큰:", tokens)
print("\\n1. Causal 마스크 생성:")
mask = create_causal_mask(seq_len)
print(mask.astype(int))
print("(1: 참조 가능, 0: 참조 불가)")

print("\\n2. Masked Attention 수행:")
output, weights = masked_attention(Q, K, V, mask)

print("\\n✅ 각 토큰은 자신과 이전 토큰만 참조!")
print("예: 'learning'(위치 3)은 I, love, machine, learning만 참조")

## AI Developed Version 1
Source: 065_Causal_Masked_Attention_#Uad6c#Ud604.ipynb

# 📘 Causal (Masked) Attention 구현

## "Attention Is All You Need" 재구현 실습 - Causal Attention

---

### 🎯 학습 목표

1. Causal Attention이 왜 필요한지 이해하기
2. Causal Mask(하삼각 행렬)를 만들고 적용하기
3. Causal Attention이 GPT 같은 모델에서 어떻게 사용되는지 이해하기

### 📋 선수 지식

- 노트북 063, 064의 Scaled Dot-Product Attention
- Padding Mask 개념 (노트북 064)

---

### 📖 배경: 왜 Causal Attention이 필요한가?

텍스트를 **생성(generation)**할 때를 생각해봅시다.

"나는 오늘 학교에 ___" 에서 빈칸을 채울 때,
빈칸 **뒤에 올 단어**를 미리 알 수는 없겠죠?

| 위치 | 볼 수 있는 토큰 | 볼 수 없는 토큰 |
|------|--------------|----------------|
| 나는 | 나는 | 오늘, 학교에, 갔다 |
| 오늘 | 나는, 오늘 | 학교에, 갔다 |
| 학교에 | 나는, 오늘, 학교에 | 갔다 |
| 갔다 | 나는, 오늘, 학교에, 갔다 | (없음) |

이렇게 **과거와 현재만 보고 미래를 차단**하는 것이 **Causal Attention**입니다.

- **Causal** = 인과적 (원인→결과 방향만 허용)
- **GPT, 텍스트 생성 모델**에서 사용됩니다
- Transformer의 **Decoder**에서 사용됩니다

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print(f"PyTorch 버전: {torch.__version__}")

## 1단계: Causal Mask 이해하기

Causal Mask는 **하삼각 행렬(lower triangular matrix)**입니다.

```
시퀀스: [A, B, C, D]

Causal Mask:
         A  B  C  D
    A  [ 1  0  0  0 ]   ← A는 A만 볼 수 있음
    B  [ 1  1  0  0 ]   ← B는 A, B만 볼 수 있음
    C  [ 1  1  1  0 ]   ← C는 A, B, C만 볼 수 있음
    D  [ 1  1  1  1 ]   ← D는 A, B, C, D 모두 볼 수 있음
```

1 = 볼 수 있음, 0 = 볼 수 없음 (미래 토큰)

In [ ]:
# ============================================================
# Causal Mask 만들기

In [ ]:
seq_len = 4

# 방법 1: torch.tril (하삼각 행렬 생성)
# tril = triangular lower (하삼각)
causal_mask_1 = torch.tril(torch.ones(seq_len, seq_len))

print("방법 1 - torch.tril 사용:")
print(causal_mask_1)
print()

# 방법 2: torch.ones + triu (상삼각을 0으로)
# triu = triangular upper (상삼각)
causal_mask_2 = 1 - torch.triu(torch.ones(seq_len, seq_len), diagonal=1)

print("방법 3 - torch.triu 활용:")
print(causal_mask_2)
print()

# 방법 3: 불리언 마스크 (True/False)
causal_mask_bool = torch.tril(torch.ones(seq_len, seq_len)).bool()

print("불리언 마스크:")
print(causal_mask_bool)
print()

# 모두 같은 결과인지 확인
print(f"방법 1 == 방법 2: {torch.equal(causal_mask_1, causal_mask_2)}")

## 2단계: Causal Mask 적용 원리

### masked_fill의 동작 원리

mask가 `0`(또는 `False`)인 위치에 `-inf`를 넣으면,
softmax를 거친 후 해당 위치의 확률이 `0`이 됩니다.

```
scores:     [2.1, 0.5, 1.8, -0.3]   (원래 attention scores)
mask:       [  1,   1,   0,    0]    (1=유효, 0=미래)
masked:     [2.1, 0.5, -inf, -inf]   (미래 위치에 -inf)
softmax:    [0.83, 0.17, 0.0, 0.0]   (미래 위치가 0이 됨!)
```

In [ ]:
# ============================================================
# masked_fill 동작 확인

In [ ]:
# 예시 scores
example_scores = torch.tensor([[2.1, 0.5, 1.8, -0.3]])
example_mask = torch.tensor([[True, True, False, False]])  # True=유효, False=마스킹

print("원래 scores:", example_scores)
print("mask:", example_mask)
print()

# mask가 False인 위치에 -inf 채우기
# ~example_mask = mask를 뒤집음 (True↔False)
masked_scores = example_scores.masked_fill(~example_mask, float('-inf'))
print("masked_fill 후:", masked_scores)

# softmax 적용
softmax_result = F.softmax(masked_scores, dim=-1)
print("softmax 결과:", softmax_result)
print()
print("[확인] 미래 위치(index 2, 3)의 확률이 0이 되었습니다!")
print(f"  유효한 위치의 합: {softmax_result[0, :2].sum():.4f} (1이어야 함)")

## 3단계: Causal Attention 구현

이제 전체 Causal Attention을 구현합니다.

In [ ]:
# ============================================================
# Causal (Masked) Attention 함수

In [ ]:
def causal_attention(Q, K, V, verbose=False):
    """
    Causal (Masked) Scaled Dot-Product Attention을 수행합니다.
    
    미래 토큰을 볼 수 없도록 causal mask를 자동으로 생성하고 적용합니다.
    
    Args:
        Q: Query, shape = (batch_size, seq_len, d_k)
        K: Key, shape = (batch_size, seq_len, d_k)  
        V: Value, shape = (batch_size, seq_len, d_v)
        verbose: 중간 과정 출력 여부
    
    Returns:
        output: (batch_size, seq_len, d_v)
        attention_weights: (batch_size, seq_len, seq_len)
    """
    d_k = K.size(-1)
    seq_len = Q.size(-2)
    
    # Step 1: Attention scores 계산 + 스케일링
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if verbose:
        print(f"[1] Scores shape: {scores.shape}")
        print(f"    Scores (스케일링 후):")
        print(f"    {scores[0]}")
    
    # Step 2: Causal mask 생성
    # 하삼각 행렬: 대각선 포함 아래만 True
    causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=Q.device)).bool()
    if verbose:
        print(f"\n[2] Causal Mask:")
        print(f"    {causal_mask.int()}")
    
    # Step 3: Mask 적용 (미래 위치에 -inf)
    # ~causal_mask: True와 False를 뒤집음
    # 즉, 미래 위치(상삼각)가 True가 되어 -inf로 채워짐
    scores = scores.masked_fill(~causal_mask, float('-inf'))
    if verbose:
        print(f"\n[3] Mask 적용 후 Scores:")
        print(f"    {scores[0]}")
    
    # Step 4: Softmax
    attention_weights = F.softmax(scores, dim=-1)
    if verbose:
        print(f"\n[4] Attention Weights (softmax 후):")
        print(f"    {attention_weights[0]}")
    
    # Step 5: Value에 가중치 적용
    output = attention_weights @ V
    if verbose:
        print(f"\n[5] Output shape: {output.shape}")
    
    return output, attention_weights


# 테스트
print("=" * 60)
print("Causal Attention 테스트")
print("=" * 60)

batch_size, seq_len, d_model = 1, 4, 8
x = torch.randn(batch_size, seq_len, d_model)

# 간단하게 Q=K=V=x로 테스트 (Self-Attention)
output, weights = causal_attention(x, x, x, verbose=True)

## 4단계: Causal Attention 시각화

일반 Attention과 Causal Attention의 차이를 시각화합니다.

In [ ]:
# ============================================================
# 일반 Attention vs Causal Attention 비교

In [ ]:
# 일반 Attention (마스크 없음)
def standard_attention(Q, K, V):
    d_k = K.size(-1)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights

output_std, weights_std = standard_attention(x, x, x)
output_causal, weights_causal = causal_attention(x, x, x)

# 시각화
tokens = ["나는", "오늘", "학교에", "갔다"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (title, w) in enumerate([
    ("일반 Attention\n(모든 토큰을 볼 수 있음)", weights_std[0].detach().numpy()),
    ("Causal Attention\n(미래 토큰을 볼 수 없음)", weights_causal[0].detach().numpy())
]):
    ax = axes[idx]
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(tokens, fontsize=11)
    ax.set_yticklabels(tokens, fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Key (정보 제공)")
    ax.set_ylabel("Query (정보 요청)")
    
    for i in range(4):
        for j in range(4):
            color = "white" if w[i, j] > 0.5 else "black"
            ax.text(j, i, f"{w[i, j]:.3f}", ha="center", va="center",
                    fontsize=10, color=color)
    plt.colorbar(im, ax=ax)

plt.suptitle("Standard vs Causal Attention", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("[관찰 포인트]")
print("  - Causal Attention에서 우상삼각 부분이 0 (미래를 볼 수 없음)")
print("  - '나는'은 자기 자신만 볼 수 있음 (weight = 1.0)")
print("  - '갔다'는 모든 토큰을 볼 수 있음 (가장 많은 정보)")

## 5단계: nn.Module로 Causal Attention 클래스 구현

실전에서 사용할 수 있는 클래스 형태로 구현합니다.

In [ ]:
# ============================================================
# Causal Self-Attention 클래스

In [ ]:
class CausalSelfAttention(nn.Module):
    """
    Causal Self-Attention 레이어.
    GPT와 같은 auto-regressive 모델의 핵심 구성요소입니다.
    
    특징:
    - 미래 토큰을 볼 수 없도록 causal mask 적용
    - nn.Linear를 사용한 Q, K, V 변환
    - 최대 시퀀스 길이를 미리 설정하여 mask를 buffer로 등록
    """
    
    def __init__(self, d_model, d_k, d_v, max_seq_len=512):
        super().__init__()
        
        self.d_k = d_k
        
        # Q, K, V 변환 레이어
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_v, bias=False)
        
        # Causal mask를 buffer로 등록
        # buffer: 학습되지 않지만 모델 저장 시 함께 저장되는 텐서
        # 매번 새로 생성하지 않아도 됨 (효율성)
        causal_mask = torch.tril(
            torch.ones(max_seq_len, max_seq_len)
        ).bool()
        self.register_buffer('mask', causal_mask)
    
    def forward(self, x):
        """
        Args:
            x: (batch_size, seq_len, d_model)
        Returns:
            output: (batch_size, seq_len, d_v)
            weights: (batch_size, seq_len, seq_len)
        """
        batch_size, seq_len, _ = x.shape
        
        # Q, K, V 생성
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)
        
        # Attention scores
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Causal mask 적용
        # self.mask[:seq_len, :seq_len]: 현재 시퀀스 길이에 맞게 슬라이싱
        mask = self.mask[:seq_len, :seq_len]
        scores = scores.masked_fill(~mask, float('-inf'))
        
        # Softmax + Value 가중 평균
        weights = F.softmax(scores, dim=-1)
        output = weights @ V
        
        return output, weights


# 테스트
d_model, d_k, d_v = 16, 16, 16
model = CausalSelfAttention(d_model, d_k, d_v)

# 배치 입력
x_test = torch.randn(2, 6, d_model)  # 배치 2, 시퀀스 길이 6
output_test, weights_test = model(x_test)

print(f"입력 shape:  {x_test.shape}")
print(f"출력 shape:  {output_test.shape}")
print(f"가중치 shape: {weights_test.shape}")
print()
print("첫 번째 문장의 Causal Attention weights:")
print(weights_test[0].detach())
print()
print("[확인] 상삼각 부분이 모두 0인지 확인하세요!")
print(f"  상삼각 최대값: {torch.triu(weights_test[0], diagonal=1).max():.10f}")
print(f"  → 0이면 미래 정보 차단 성공! ✅")

## 6단계: Causal Attention의 Auto-Regressive 동작

### Auto-Regressive란?

텍스트 생성 시, 한 토큰씩 순차적으로 생성하는 방식입니다.

```
Step 1: [나는]           → "오늘" 예측
Step 2: [나는, 오늘]      → "학교에" 예측
Step 3: [나는, 오늘, 학교에] → "갔다" 예측
```

Causal Attention 덕분에, 학습 시에는 한 번에 전체 시퀀스를 넣고
각 위치에서 미래를 못 보게 해서 효율적으로 학습할 수 있습니다.

In [ ]:
# ============================================================
# Auto-Regressive 동작 시뮬레이션

In [ ]:
# 전체 시퀀스를 한 번에 처리하는 것과
# 한 토큰씩 늘려가며 처리하는 것이 동일한 결과를 내는지 확인합니다.

d_model_ar = 8
model_ar = CausalSelfAttention(d_model_ar, d_model_ar, d_model_ar)
model_ar.eval()  # 평가 모드 (dropout 등 비활성화)

# 4개 토큰으로 구성된 시퀀스
full_sequence = torch.randn(1, 4, d_model_ar)

# 방법 1: 전체 시퀀스를 한 번에 넣기 (Causal Mask 사용)
full_output, _ = model_ar(full_sequence)

# 방법 2: 한 토큰씩 늘려가며 처리
print("=== Auto-Regressive 동작 확인 ===")
print()

for t in range(1, 5):  # 1, 2, 3, 4 토큰
    partial_sequence = full_sequence[:, :t, :]
    partial_output, _ = model_ar(partial_sequence)
    
    # 마지막 토큰의 출력 비교
    diff = (full_output[0, t-1] - partial_output[0, t-1]).abs().max()
    print(f"토큰 {t}개: partial[-1] == full[{t-1}] ? 차이={diff:.10f} {'✅' if diff < 1e-6 else '❌'}")

print()
print("[핵심 인사이트]")
print("  Causal Mask 덕분에, 전체 시퀀스를 한 번에 넣어도")
print("  각 위치의 출력은 해당 위치까지만 본 것과 동일합니다!")
print("  → 학습 시 효율적 (한 번의 forward pass로 모든 위치 학습)")

## 7단계: Padding Mask + Causal Mask 결합

실전에서는 Padding Mask와 Causal Mask를 **동시에** 적용해야 합니다.
- Causal Mask: 미래 토큰 차단
- Padding Mask: PAD 토큰 무시

두 마스크를 결합하면 됩니다!

In [ ]:
# ============================================================
# Padding + Causal Mask 결합

In [ ]:
seq_len_combined = 5

# 시나리오: 5개 위치 중 3개만 유효 (나머지 2개는 PAD)
# 문장: ["나는", "학생", "이다", "[PAD]", "[PAD]"]

# Padding mask: 1=유효, 0=PAD
padding_mask = torch.tensor([1, 1, 1, 0, 0]).float()

# Causal mask: 하삼각 행렬
causal_mask = torch.tril(torch.ones(seq_len_combined, seq_len_combined))

# 결합: 두 mask의 AND 연산 (element-wise 곱)
# padding_mask를 (1, seq_len) → (seq_len, seq_len)로 브로드캐스트
combined_mask = causal_mask * padding_mask.unsqueeze(0)

print("Causal Mask:")
print(causal_mask.int())
print()

print("Padding Mask (브로드캐스트):")
print(padding_mask.unsqueeze(0).expand(seq_len_combined, -1).int())
print()

print("결합된 Mask (Causal × Padding):")
print(combined_mask.int())
print()

labels = ["나는", "학생", "이다", "[PAD]", "[PAD]"]

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for idx, (title, m) in enumerate([
    ("Causal Mask", causal_mask),
    ("Padding Mask", padding_mask.unsqueeze(0).expand(5, -1)),
    ("결합 Mask", combined_mask)
]):
    ax = axes[idx]
    im = ax.imshow(m.numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    ax.set_xticklabels(labels, rotation=45, fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_title(title, fontsize=12)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{int(m[i,j])}", ha="center", va="center", fontsize=10)

plt.tight_layout()
plt.show()

print("[결합 Mask 해석]")
print("  - 하삼각 중에서도 PAD 열은 0 → PAD 토큰 무시")
print("  - 상삼각은 여전히 0 → 미래 토큰 차단")
print("  - 두 가지 제약이 모두 적용됩니다!")

## 🔑 핵심 정리

### Causal Attention의 핵심

| 특성 | 설명 |
|------|------|
| **목적** | 미래 토큰의 정보 차단 |
| **사용처** | GPT, Decoder, 텍스트 생성 |
| **구현** | 하삼각 행렬(tril) → masked_fill(-inf) → softmax |
| **효과** | 학습 시 효율적 병렬 처리 가능 |

### Encoder vs Decoder의 Attention 차이

| | Encoder | Decoder |
|--|---------|--------|
| **Mask** | Padding만 | Padding + Causal |
| **방향** | 양방향 (Bi-directional) | 단방향 (Uni-directional) |
| **대표 모델** | BERT | GPT |

### 다음 단계

- **노트북 066**: Multi-Head Attention 완전 구현

## AI Developed Version 2
Source: 03_Causal_Masked_Attention.ipynb

# 📚 Notebook 3: Causal (Masked) Attention 구현

## "Attention Is All You Need" 재구현 - Decoder의 Masked Self-Attention

---

### 🎯 학습 목표
Decoder에서 사용하는 **Causal Attention** (= Masked Self-Attention)을 이해하고 구현합니다.

### 💡 왜 Causal Attention이 필요한가?

텍스트를 생성할 때, 모델은 **아직 생성하지 않은 미래 단어를 볼 수 없어야** 합니다.

```
예: "나는 고양이를 좋아한다" 생성 중

단계 1: "나는" 생성       → 참고 가능: [나는]
단계 2: "고양이를" 생성    → 참고 가능: [나는, 고양이를]
단계 3: "좋아" 생성       → 참고 가능: [나는, 고양이를, 좋아]
단계 4: "한다" 생성       → 참고 가능: [나는, 고양이를, 좋아, 한다]
```

### 📋 목차
1. Causal Mask가 필요한 이유 (시각적 설명)
2. 삼각행렬(Lower Triangular Matrix) 이해
3. Causal Mask 생성
4. Causal Attention 구현
5. 일반 Attention과 Causal Attention 비교
6. 자동 회귀(Autoregressive) 생성 시뮬레이션

---

In [ ]:
# ============================================================
# 환경 설정

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print("✅ 라이브러리 로딩 완료!")

In [ ]:
# ============================================================
# 이전 노트북의 Attention 함수 (복습)

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    Attention(Q,K,V) = softmax(QK^T / √d_k) V
    
    mask: 1 = 주목 가능, 0 = 주목 불가 (마스킹)
    """
    d_k = K.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    if mask is not None:
        # mask가 0인 위치 → -inf → softmax 후 0이 됨
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)
    return output, attention_weights

print("✅ Attention 함수 준비 완료")

## 1. Causal Mask란? 시각적으로 이해하기 🎭

Causal Mask는 **하삼각행렬(Lower Triangular Matrix)** 형태입니다.

```
문장: [A, B, C, D]

      Key:  A  B  C  D
Query A  [  1  0  0  0 ]   ← A는 자신만 볼 수 있음
Query B  [  1  1  0  0 ]   ← B는 A, B를 볼 수 있음
Query C  [  1  1  1  0 ]   ← C는 A, B, C를 볼 수 있음
Query D  [  1  1  1  1 ]   ← D는 모두 볼 수 있음
```

- **1** = 볼 수 있음 (Attention 허용)
- **0** = 볼 수 없음 (미래 토큰이므로 차단)

In [ ]:
# ============================================================
# Causal Mask 생성 방법

In [ ]:
seq_len = 5  # 시퀀스 길이

# 방법 1: torch.tril (하삼각 행렬)
# tril = "triangular lower" = 아래쪽 삼각형
causal_mask_1 = torch.tril(torch.ones(seq_len, seq_len))

print("📐 방법 1: torch.tril 사용")
print(causal_mask_1)

# 방법 2: torch.ones + torch.triu 로 상삼각을 0으로
causal_mask_2 = torch.ones(seq_len, seq_len)
causal_mask_2 = torch.tril(causal_mask_2)

print(f"\n📐 방법 2: 결과 동일함")
print(causal_mask_2)

# 두 방법이 같은지 확인
print(f"\n두 방법의 결과가 같은가? {torch.equal(causal_mask_1, causal_mask_2)}")

In [ ]:
# ============================================================
# Causal Mask 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

tokens = ['나는', '고양이를', '좋아', '한다', 'EOS']

# Causal Mask
axes[0].imshow(causal_mask_1.numpy(), cmap='RdYlGn', vmin=0, vmax=1)
axes[0].set_xticks(range(seq_len))
axes[0].set_yticks(range(seq_len))
axes[0].set_xticklabels(tokens, fontsize=10)
axes[0].set_yticklabels(tokens, fontsize=10)
axes[0].set_title('Causal Mask\n(초록=허용, 빨강=차단)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Key (대상)')
axes[0].set_ylabel('Query (주체)')
for i in range(seq_len):
    for j in range(seq_len):
        val = int(causal_mask_1[i, j].item())
        axes[0].text(j, i, str(val), ha='center', va='center', 
                    fontsize=14, fontweight='bold',
                    color='white' if val == 0 else 'black')

# 일반 Attention (마스크 없음)
no_mask = torch.ones(seq_len, seq_len)
axes[1].imshow(no_mask.numpy(), cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_xticks(range(seq_len))
axes[1].set_yticks(range(seq_len))
axes[1].set_xticklabels(tokens, fontsize=10)
axes[1].set_yticklabels(tokens, fontsize=10)
axes[1].set_title('일반 Attention (마스크 없음)\n(모든 위치에 주목 가능)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Key (대상)')
axes[1].set_ylabel('Query (주체)')
for i in range(seq_len):
    for j in range(seq_len):
        axes[1].text(j, i, '1', ha='center', va='center', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("💡 Causal Mask: 대각선 아래만 1 → 과거와 현재만 볼 수 있음!")

## 2. Causal Attention 구현 및 비교

이제 실제로 Causal Mask를 Attention에 적용해 봅시다.

In [ ]:
# ============================================================
# Causal Attention vs 일반 Attention 비교

In [ ]:
d_k = 8
d_v = 8

# Q, K, V 생성
Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_v)

# 1) 일반 Attention (마스크 없음)
output_normal, weights_normal = scaled_dot_product_attention(Q, K, V)

# 2) Causal Attention (마스크 적용)
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
output_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print("📊 Attention 가중치 비교")
print("=" * 60)

print("\n❌ 일반 Attention (모든 위치에 주목 가능):")
print("  각 행의 합:", [f"{weights_normal[i].sum():.3f}" for i in range(seq_len)])
print(weights_normal.detach().numpy().round(3))

print("\n✅ Causal Attention (미래 토큰 차단):")
print("  각 행의 합:", [f"{weights_causal[i].sum():.3f}" for i in range(seq_len)])
print(weights_causal.detach().numpy().round(3))

print("\n💡 Causal Attention에서 상삼각(오른쪽 위) 부분이 0인 것을 확인!")
print("   각 행의 합은 여전히 1.0 (softmax가 허용된 위치에서만 확률 분배)")

In [ ]:
# ============================================================
# 가중치 시각화 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 일반 Attention
im1 = axes[0].imshow(weights_normal.detach().numpy(), cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('일반 Attention\n(마스크 없음)', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(seq_len))
axes[0].set_yticks(range(seq_len))
axes[0].set_xticklabels(tokens)
axes[0].set_yticklabels(tokens)
for i in range(seq_len):
    for j in range(seq_len):
        v = weights_normal[i,j].item()
        axes[0].text(j, i, f'{v:.2f}', ha='center', va='center',
                    color='white' if v > 0.4 else 'black', fontsize=9)

# Causal Attention
im2 = axes[1].imshow(weights_causal.detach().numpy(), cmap='Oranges', vmin=0, vmax=1)
axes[1].set_title('Causal (Masked) Attention\n(미래 토큰 차단)', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(seq_len))
axes[1].set_yticks(range(seq_len))
axes[1].set_xticklabels(tokens)
axes[1].set_yticklabels(tokens)
for i in range(seq_len):
    for j in range(seq_len):
        v = weights_causal[i,j].item()
        axes[1].text(j, i, f'{v:.2f}', ha='center', va='center',
                    color='white' if v > 0.4 else 'black', fontsize=9)

plt.tight_layout()
plt.show()

## 3. 마스킹 과정을 단계별로 살펴보기 🔍

마스킹이 내부적으로 어떻게 동작하는지 한 단계씩 확인해봅시다.

In [ ]:
# ============================================================
# 마스킹 과정 상세 분석

In [ ]:
# 간단한 3x3 예시
simple_seq = 3
Q_s = torch.randn(simple_seq, 4)
K_s = torch.randn(simple_seq, 4)

# Step 1: 유사도 점수 계산
scores = torch.matmul(Q_s, K_s.T) / math.sqrt(4)
print("Step 1: 스케일링된 유사도 점수")
print(scores.detach().numpy().round(3))

# Step 2: Causal Mask 생성
mask = torch.tril(torch.ones(simple_seq, simple_seq))
print(f"\nStep 2: Causal Mask")
print(mask.numpy().astype(int))

# Step 3: 마스킹 적용 (0인 위치를 -inf로)
masked_scores = scores.masked_fill(mask == 0, float('-inf'))
print(f"\nStep 3: 마스킹 적용 후 (0 → -inf)")
print("  미래 위치가 -inf로 바뀐 것을 확인!")
# -inf 표시를 위해 수동 출력
for i in range(simple_seq):
    row = []
    for j in range(simple_seq):
        val = masked_scores[i, j].item()
        if val == float('-inf'):
            row.append('  -inf')
        else:
            row.append(f'{val:6.3f}')
    print(f"  [{', '.join(row)}]")

# Step 4: Softmax 적용
weights = F.softmax(masked_scores, dim=-1)
print(f"\nStep 4: Softmax 적용 후")
print("  -inf → 0.000 으로 변환됨!")
print(weights.detach().numpy().round(3))

print(f"\n💡 핵심: -inf에 e^(-inf) = 0 이므로 softmax 후 가중치가 0이 됩니다!")

## 4. 자동 회귀(Autoregressive) 생성 시뮬레이션 🤖

GPT 같은 모델이 텍스트를 생성하는 과정을 시뮬레이션해봅시다.

자동 회귀(Autoregressive)란?
- 한 번에 한 토큰씩 순서대로 생성
- 이전에 생성한 토큰만 참고하여 다음 토큰 예측
- Causal Attention이 이를 가능하게 함

In [ ]:
# ============================================================
# 자동 회귀 생성 시뮬레이션

In [ ]:
print("🤖 자동 회귀 생성 시뮬레이션")
print("=" * 60)

# 전체 문장의 토큰
full_tokens = ['<BOS>', '나는', '고양이를', '좋아', '한다']
full_seq_len = len(full_tokens)

# 전체 시퀀스에 대한 Q, K, V
d_k = 8
Q_full = torch.randn(full_seq_len, d_k)
K_full = torch.randn(full_seq_len, d_k)
V_full = torch.randn(full_seq_len, d_k)

# 각 생성 단계에서의 Attention 시각화
fig, axes = plt.subplots(1, full_seq_len, figsize=(20, 4))

for step in range(full_seq_len):
    # 현재까지 생성된 토큰만 사용
    current_len = step + 1
    Q_step = Q_full[:current_len]
    K_step = K_full[:current_len]
    V_step = V_full[:current_len]
    
    # Causal Mask (현재까지만)
    mask_step = torch.tril(torch.ones(current_len, current_len))
    _, w_step = scaled_dot_product_attention(Q_step, K_step, V_step, mask=mask_step)
    
    # 시각화 (5x5 크기에 맞춰서)
    display_weights = torch.zeros(full_seq_len, full_seq_len)
    display_weights[:current_len, :current_len] = w_step.detach()
    
    axes[step].imshow(display_weights.numpy(), cmap='Blues', vmin=0, vmax=1)
    axes[step].set_title(f'Step {step+1}\n생성: "{full_tokens[step]}"', fontsize=10)
    axes[step].set_xticks(range(full_seq_len))
    axes[step].set_yticks(range(full_seq_len))
    axes[step].set_xticklabels(full_tokens, fontsize=7, rotation=45)
    axes[step].set_yticklabels(full_tokens, fontsize=7)
    
    # 현재까지 주목 가능한 영역 표시
    current_tokens = full_tokens[:current_len]
    print(f"Step {step+1}: '{full_tokens[step]}' 생성 → 참고 가능: {current_tokens}")

plt.suptitle('자동 회귀 생성 과정에서의 Attention', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Causal Attention + Padding Mask 결합 🎭

실제 모델에서는 Causal Mask와 Padding Mask를 **동시에** 적용해야 합니다.

```
최종 마스크 = Causal Mask AND Padding Mask
(둘 다 1인 경우에만 주목 허용)
```

In [ ]:
# ============================================================
# Causal Mask + Padding Mask 결합

In [ ]:
seq_len = 5

# Causal Mask (미래 차단)
causal = torch.tril(torch.ones(seq_len, seq_len))

# Padding Mask (마지막 2개가 PAD)
# 실제 토큰: [나는, 고양이를, 좋아한다, PAD, PAD]
padding = torch.tensor([1, 1, 1, 0, 0]).float()  # 1=실제, 0=PAD
padding_2d = padding.unsqueeze(0).expand(seq_len, -1)  # (5,) → (5, 5)

# 결합: 두 마스크를 곱하면 됨 (AND 연산)
combined_mask = causal * padding_2d

print("📐 각 마스크와 결합 결과")
print(f"\nCausal Mask:")
print(causal.numpy().astype(int))
print(f"\nPadding Mask (2D):")
print(padding_2d.numpy().astype(int))
print(f"\n결합 Mask (Causal × Padding):")
print(combined_mask.numpy().astype(int))

labels = ['나는', '고양이를', '좋아한다', 'PAD', 'PAD']

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, m, title in zip(axes, 
                         [causal, padding_2d, combined_mask],
                         ['Causal Mask', 'Padding Mask', 'Combined Mask']):
    ax.imshow(m.numpy(), cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(labels, fontsize=9, rotation=30)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_title(title, fontsize=14, fontweight='bold')
    for i in range(seq_len):
        for j in range(seq_len):
            ax.text(j, i, str(int(m[i,j].item())), ha='center', va='center',
                   fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("💡 Combined Mask: 미래 토큰도 차단하고, PAD 토큰도 차단!")

In [ ]:
# ============================================================
# 결합 마스크를 사용한 Attention 계산

In [ ]:
Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_v)

output, weights = scaled_dot_product_attention(Q, K, V, mask=combined_mask)

print("Combined Mask 적용 후 Attention 가중치:")
for i, label in enumerate(labels):
    w = weights[i].detach().numpy()
    parts = [f"{labels[j]}({w[j]:.3f})" for j in range(seq_len)]
    print(f"  {label} → {', '.join(parts)}")

print(f"\n✅ PAD 토큰의 가중치가 0이고, 미래 토큰도 차단된 것을 확인!")

## 📝 핵심 정리

| 마스크 종류 | 목적 | 사용 위치 |
|-------------|------|----------|
| **Padding Mask** | PAD 토큰 무시 | Encoder & Decoder |
| **Causal Mask** | 미래 토큰 차단 | Decoder의 Self-Attention |
| **Combined** | 둘 다 적용 | Decoder의 Self-Attention |

### 구현 핵심
1. `torch.tril()`로 하삼각 행렬 생성
2. `masked_fill(mask==0, -inf)`로 차단 위치에 -inf
3. `softmax` 후 -inf → 0이 되어 자연스럽게 무시됨

### 다음 노트북 예고 🔜
Notebook 4에서는 **Multi-Head Attention**을 구현합니다.
→ 여러 개의 Attention을 병렬로 수행하여 다양한 관점에서 정보를 수집!

## AI Developed Version 3
Source: 065_Causal_Masked_Attention.ipynb

# 📘 실습 065: Causal (Masked) Attention 구현

## 🎯 이번 실습의 핵심 질문

> **"문장을 생성할 때, 아직 생성되지 않은 미래 단어를 미리 볼 수 있을까?"**

당연히 **안 됩니다!** 실제 언어 모델(GPT 등)은 단어를 하나씩 순서대로 생성합니다.

### 예시:
```
입력: "나는 사과를"
모델: → "먹었다" 생성

"먹었다"를 예측할 때:
  ✅ "나는" 참고 가능   (이미 생성된 단어)
  ✅ "사과를" 참고 가능 (이미 생성된 단어)
  ❌ "먹었다" 자기 자신은 미래 → 참고 불가 (치팅!)
```

### 이를 구현하는 방법: **Causal Mask (인과 마스크)**

미래 위치를 `-inf`로 마스킹 → softmax 후 0이 됨 → 미래 무시!

---

### 학습 목표
1. Causal Mask의 개념과 필요성 이해
2. 삼각 행렬(Triangular Matrix)로 마스크 생성
3. Masked Self-Attention 구현
4. 실제 언어 모델에서 사용 방식 이해

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

torch.manual_seed(42)
print("✅ 라이브러리 로드 완료")

## 🔺 Step 1: Causal Mask란?

아래 표를 봅시다. 문장 "나는 사과를 먹었다"를 처리할 때:

```
       나는  사과를 먹었다
나는  [ ✅    ❌     ❌  ]   ← "나는"은 자기 자신만 봄
사과를[ ✅    ✅     ❌  ]   ← "사과를"은 "나는", "사과를" 봄
먹었다[ ✅    ✅     ✅  ]   ← "먹었다"는 모두 봄
```

이것을 수학적으로 표현하면 **하삼각 행렬(Lower Triangular Matrix)**!

```
마스크 (1=무시, 0=허용):
[ 0  1  1 ]
[ 0  0  1 ]
[ 0  0  0 ]
```

In [ ]:
# ============================================================
# 🔺 Step 1: Causal Mask 생성

In [ ]:
def create_causal_mask(seq_len):
    """
    Causal (자기회귀) 마스크 생성
    
    미래 위치는 1 (무시), 과거/현재는 0 (허용)
    → 상삼각 행렬(대각선 위)이 1인 마스크
    
    Args:
        seq_len: 시퀀스 길이
    
    Returns:
        mask: shape = (seq_len, seq_len)
              [i][j] = 1이면 i번째 토큰이 j번째 토큰을 볼 수 없음
    """
    # torch.triu: 상삼각 행렬 (diagonal=1: 대각선 위부터)
    # diagonal=0이면 대각선 포함, diagonal=1이면 대각선 제외
    # 
    # 예 (seq_len=4):
    # triu(ones, diagonal=1):
    # [0 1 1 1]
    # [0 0 1 1]
    # [0 0 0 1]
    # [0 0 0 0]
    #
    # 1인 위치 = 미래 = 마스킹 (무시)
    
    mask = torch.triu(
        torch.ones(seq_len, seq_len, dtype=torch.long),
        diagonal=1  # 대각선은 자기 자신 → 허용 (0)
    )
    return mask


# 테스트
words = ["나는", "사과를", "먹었다", "."]  # 4개 토큰
seq_len = len(words)
causal_mask = create_causal_mask(seq_len)

print("=" * 60)
print(f"Causal Mask (seq_len={seq_len})")
print("=" * 60)
print("\n마스크 행렬 (0=허용, 1=마스킹):")
print(causal_mask.numpy())
print()

# 직관적 표현
print("직관적 표현 (✅=볼 수 있음, ❌=볼 수 없음):")
header = "         " + "".join(f"{w:>8}" for w in words)
print(header)
print("-" * (9 + 8 * seq_len))
for i, word_i in enumerate(words):
    row = f"{word_i:>8} |" 
    for j in range(seq_len):
        if causal_mask[i][j] == 0:
            row += "       ✅"
        else:
            row += "       ❌"
    print(row)

print("\n💡 각 행(row) = '이 단어가 볼 수 있는 단어들'")
print("   → 자기 자신과 과거 단어만 볼 수 있음!")

In [ ]:
# ============================================================
# 📊 Step 2: Causal Mask 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: Causal 마스크 자체
ax1 = axes[0]
mask_display = np.where(causal_mask.numpy() == 0, 1, 0)  # 0/1 반전 (시각화용)
sns.heatmap(
    mask_display,
    annot=True,
    fmt='d',
    cmap='RdYlGn',
    xticklabels=words,
    yticklabels=words,
    ax=ax1,
    vmin=0, vmax=1,
    linewidths=0.5,
    linecolor='gray'
)
ax1.set_title('Causal Mask\n(초록=볼 수 있음, 빨강=볼 수 없음)', fontsize=12)
ax1.set_xlabel('Key (볼 수 있는 대상)')
ax1.set_ylabel('Query (현재 처리 중인 단어)')

# 오른쪽: Causal Mask 적용 후 Attention Weight
d_k = 8
torch.manual_seed(42)
Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

# 마스크 적용 전 scores
scores = Q @ K.T / math.sqrt(d_k)

# 마스크 적용: 미래 위치에 -inf
scores_masked = scores.masked_fill(causal_mask == 1, float('-inf'))

# Softmax → Attention Weight
attn_weights = F.softmax(scores_masked, dim=-1)

ax2 = axes[1]
sns.heatmap(
    attn_weights.detach().numpy(),
    annot=True,
    fmt='.3f',
    cmap='Blues',
    xticklabels=words,
    yticklabels=words,
    ax=ax2,
    vmin=0, vmax=1
)
ax2.set_title('Causal Attention Weights\n(상삼각 = 0, 미래 참조 불가)', fontsize=12)
ax2.set_xlabel('Key')
ax2.set_ylabel('Query')

plt.tight_layout()
plt.savefig('causal_attention.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n검증: 상삼각 부분이 모두 0인가?")
upper_triangle = attn_weights[causal_mask == 1]
print(f"  마스킹된 위치의 Attention Weight: {upper_triangle.tolist()}")
print(f"  ✅ 모두 0! (NaN이 있으면 -inf → softmax → 0)")

In [ ]:
# ============================================================
# 🏗️ Step 3: Masked Self-Attention 클래스 구현

In [ ]:
class CausalSelfAttention(nn.Module):
    """
    Causal (Masked) Self-Attention
    GPT 같은 자기회귀 언어 모델에서 사용
    
    핵심:
    - 현재 위치에서 과거(자신 포함)만 볼 수 있음
    - 미래 위치는 마스크로 차단
    """
    
    def __init__(self, d_model, d_k, max_seq_len=512):
        """
        Args:
            d_model:     입력 임베딩 차원
            d_k:         Query/Key/Value 차원
            max_seq_len: 처리할 수 있는 최대 시퀀스 길이
        """
        super(CausalSelfAttention, self).__init__()
        
        self.d_k = d_k
        
        # Q, K, V 변환 레이어
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_k, bias=False)
        
        # 출력 변환 레이어
        self.out_proj = nn.Linear(d_k, d_model)
        
        # ⚡ 효율성 핵심: 마스크를 미리 계산해서 버퍼로 저장
        # register_buffer: 학습 파라미터는 아니지만 모델과 함께 저장/이동
        # (model.to(device) 할 때 자동으로 같이 이동)
        causal_mask = torch.triu(
            torch.ones(max_seq_len, max_seq_len, dtype=torch.bool),
            diagonal=1
        )
        self.register_buffer('causal_mask', causal_mask)
    
    def forward(self, x):
        """
        Args:
            x: 입력  shape = (batch, seq_len, d_model)
        
        Returns:
            output:  shape = (batch, seq_len, d_model)
            weights: shape = (batch, seq_len, seq_len)
        """
        batch_size, seq_len, _ = x.shape
        
        # 1. Q, K, V 계산
        Q = self.W_Q(x)   # (batch, seq_len, d_k)
        K = self.W_K(x)   # (batch, seq_len, d_k)
        V = self.W_V(x)   # (batch, seq_len, d_k)
        
        # 2. Attention Score 계산
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # scores shape: (batch, seq_len, seq_len)
        
        # 3. Causal Mask 적용
        # 현재 시퀀스 길이에 맞게 마스크 자르기
        # (max_seq_len 크기로 저장된 마스크에서 실제 사용할 부분만)
        mask = self.causal_mask[:seq_len, :seq_len]  # (seq_len, seq_len)
        scores = scores.masked_fill(mask, float('-inf'))
        
        # 4. Softmax
        weights = F.softmax(scores, dim=-1)
        
        # 5. Value 가중 평균
        context = torch.matmul(weights, V)  # (batch, seq_len, d_k)
        
        # 6. 출력 변환 (d_k → d_model)
        output = self.out_proj(context)     # (batch, seq_len, d_model)
        
        return output, weights


# 테스트
d_model = 64
d_k = 32
batch_size = 2
seq_len = 6

causal_attn = CausalSelfAttention(d_model=d_model, d_k=d_k, max_seq_len=128)

x = torch.randn(batch_size, seq_len, d_model)
output, weights = causal_attn(x)

print("=" * 60)
print("CausalSelfAttention 테스트")
print("=" * 60)
print(f"입력: {x.shape}")
print(f"출력: {output.shape}")
print(f"가중치: {weights.shape}")
print()

# 검증: 미래 위치가 0인지 확인
mask_check = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
future_weights = weights[0][mask_check]  # 상삼각 부분
print(f"미래 위치 Attention Weights (모두 0이어야 함):")
print(f"  최대값: {future_weights.max().item():.2e}")
print(f"  ✅ 미래 참조 완전 차단!")

In [ ]:
# ============================================================
# 🤖 Step 4: 간단한 언어 모델 시뮬레이션

In [ ]:
#
# Causal Attention이 실제로 어떻게 사용되는지 시뮬레이션
# (GPT 스타일의 자기회귀 언어 모델)
#
# 실제 학습 없이, 구조만 보여드립니다.

class TinyGPT(nn.Module):
    """
    아주 작은 GPT 스타일 언어 모델
    Causal Self-Attention을 사용하는 자기회귀 모델
    
    구조:
    입력 토큰 ID → 임베딩 → Causal Attention → LayerNorm → 출력 확률
    """
    
    def __init__(self, vocab_size, d_model, d_k, max_len):
        """
        Args:
            vocab_size: 어휘 크기 (토큰의 종류 수)
            d_model:    임베딩/모델 차원
            d_k:        Attention 차원
            max_len:    최대 시퀀스 길이
        """
        super(TinyGPT, self).__init__()
        
        # 토큰 임베딩: 각 토큰 ID → d_model 차원 벡터
        self.token_emb = nn.Embedding(vocab_size, d_model)
        
        # 위치 임베딩: 각 위치 → d_model 차원 벡터
        # (나중에 065에서 Positional Encoding으로 대체)
        self.pos_emb = nn.Embedding(max_len, d_model)
        
        # Causal Self-Attention 레이어
        self.attention = CausalSelfAttention(d_model, d_k, max_len)
        
        # Layer Normalization: 각 토큰의 표현을 정규화
        # → 학습 안정성 향상
        self.layer_norm = nn.LayerNorm(d_model)
        
        # 최종 출력 레이어: d_model → vocab_size (각 토큰의 확률)
        self.lm_head = nn.Linear(d_model, vocab_size)
    
    def forward(self, token_ids):
        """
        Args:
            token_ids: shape = (batch, seq_len)
                       각 토큰의 ID 번호
        
        Returns:
            logits: shape = (batch, seq_len, vocab_size)
                    각 위치에서 다음 토큰의 로그 확률
        """
        batch, seq_len = token_ids.shape
        
        # 위치 인덱스 생성: [0, 1, 2, ..., seq_len-1]
        positions = torch.arange(seq_len, device=token_ids.device)
        
        # 임베딩: 토큰 임베딩 + 위치 임베딩
        x = self.token_emb(token_ids) + self.pos_emb(positions)
        # shape: (batch, seq_len, d_model)
        
        # Causal Self-Attention (과거만 참조)
        attn_output, attn_weights = self.attention(x)
        
        # Residual Connection + Layer Normalization
        # Residual: x + attn_output (원래 입력을 더해서 gradient vanishing 방지)
        x = self.layer_norm(x + attn_output)
        
        # 각 위치에서 다음 토큰 예측
        logits = self.lm_head(x)
        
        return logits, attn_weights


# 간단한 어휘 사전
vocab = {
    '<PAD>': 0, '<BOS>': 1, '<EOS>': 2,
    '나는': 3, '사과를': 4, '먹었다': 5,
    '고양이': 6, '는': 7, '귀엽다': 8,
    '오늘': 9, '날씨가': 10, '좋다': 11
}
vocab_size = len(vocab)
idx_to_word = {v: k for k, v in vocab.items()}

# 모델 생성
model = TinyGPT(vocab_size=vocab_size, d_model=32, d_k=16, max_len=20)

# 예시 입력: "나는 사과를 먹었다"
input_tokens = torch.tensor([[vocab['<BOS>'], vocab['나는'], vocab['사과를'], vocab['먹었다']]])

model.eval()
with torch.no_grad():
    logits, attn_weights = model(input_tokens)

print("=" * 60)
print("TinyGPT 언어 모델 테스트")
print("=" * 60)
print(f"\n입력 토큰: {[idx_to_word[i.item()] for i in input_tokens[0]]}")
print(f"입력 shape: {input_tokens.shape}")
print(f"출력 logits shape: {logits.shape}")
print(f"  → (batch=1, seq=4, vocab={vocab_size})")
print()
print("마지막 위치('먹었다')에서 다음 토큰 예측:")
last_logits = logits[0, -1, :]   # 마지막 토큰의 로그 확률
probs = F.softmax(last_logits, dim=-1)
top_k = torch.topk(probs, k=3)
for prob, idx in zip(top_k.values, top_k.indices):
    word = idx_to_word[idx.item()]
    print(f"  '{word}': {prob.item():.4f} ({prob.item()*100:.1f}%)")
print("\n(학습 전이라 의미 없는 확률이지만, 구조는 정확!)")

In [ ]:
# ============================================================
# 📊 Step 5: Causal Attention 가중치 시각화

In [ ]:
# 4개 토큰에 대한 Causal Attention 시각화
token_names = ['<BOS>', '나는', '사과를', '먹었다']

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    attn_weights[0].detach().numpy(),
    annot=True,
    fmt='.3f',
    cmap='Blues',
    xticklabels=token_names,
    yticklabels=token_names,
    ax=ax,
    vmin=0, vmax=1,
    linewidths=0.5
)
ax.set_title('TinyGPT Causal Attention Weights\n(상삼각=0, 미래 차단!)', fontsize=12, fontweight='bold')
ax.set_xlabel('볼 수 있는 토큰 (Key)')
ax.set_ylabel('현재 처리 중인 토큰 (Query)')

# 차단된 영역 표시
for i in range(len(token_names)):
    for j in range(i+1, len(token_names)):
        ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=True, color='red', alpha=0.2))

plt.tight_layout()
plt.savefig('causal_attn_gpt.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n📖 읽는 법:")
print("  각 행 = 해당 토큰이 각 위치에 주목하는 비율")
print("  빨간 영역 = 미래 토큰 (차단됨, 항상 0)")
print("  '먹었다'는 '<BOS>', '나는', '사과를', '먹었다' 모두 볼 수 있음")
print("  '<BOS>'는 자기 자신만 볼 수 있음")

In [ ]:
# ============================================================
# 🔄 Step 6: Self-Attention vs Causal Attention 비교

In [ ]:
print("=" * 70)
print("Self-Attention vs Causal Attention 비교")
print("=" * 70)

comparison_data = [
    ("마스킹", "없음 (모든 위치 참조)", "있음 (미래 위치 차단)"),
    ("사용 모델", "BERT, 인코더", "GPT, 디코더"),
    ("방향성", "양방향 (←→)", "단방향 (→만 가능)"),
    ("학습 방식", "Masked LM (빈 칸 채우기)", "다음 토큰 예측"),
    ("예시 태스크", "문장 분류, 감정 분석", "텍스트 생성, 번역"),
    ("Attention 행렬", "대칭적 (모두 참조 가능)", "하삼각 (위만 0)"),
]

header = f"{'항목':<15} | {'Self-Attention':<30} | {'Causal Attention':<30}"
print(header)
print("-" * len(header))
for item, sa, ca in comparison_data:
    print(f"{item:<15} | {sa:<30} | {ca:<30}")

print()
print("💡 핵심 차이:")
print("  BERT = 양방향 → 문장 전체를 보고 이해 (이해 태스크에 강함)")
print("  GPT  = 단방향 → 순서대로 생성 (생성 태스크에 강함)")

## 📋 정리

### ✅ 이번 실습에서 배운 것

1. **Causal Mask의 필요성**
   - 자기회귀 언어 모델은 미래를 볼 수 없어야 함
   - 상삼각 행렬(triu)로 미래 위치 마스킹

2. **마스킹 구현**
   - `-inf`로 마스킹 → softmax 후 0
   - `torch.triu(ones, diagonal=1)` 로 마스크 생성

3. **TinyGPT 구현**
   - Causal Attention을 이용한 자기회귀 언어 모델
   - 토큰 임베딩 + 위치 임베딩 + Causal Attention

4. **Self-Attention vs Causal Attention**
   - BERT 스타일 vs GPT 스타일

### ➡️ 다음 실습 (066)
- **Multi-Head Attention** - 여러 개의 Attention을 병렬로!
- 각 헤드가 다른 관계를 학습하여 더 풍부한 표현 가능